# Faster R-CNN — Towards Real-Time Object Detection with Region Proposal Networks (2015)

_Papers · Computer Vision_

**Make region proposals nearly free: a tiny conv net (the RPN) slides over the shared feature map and predicts object boxes directly, replacing the slow Selective Search step.**

---

This notebook builds the Region Proposal Network from scratch, one piece at a time: anchor boxes, the RPN head, box decoding, and the k=9 vs k=1 anchor ablation. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build the RPN piece by piece, then test why it needs multiple anchors. In four steps: (1) decode one anchor + delta, (2) place anchors over the feature map and define the RPN head, (3) vectorized box decoding, and (4) the k=9 vs k=1 anchor ablation.

### Step 1 — Decode one anchor + delta (inverse of Eq 2)

The RPN predicts a **delta** `(tx, ty, tw, th)` relative to a fixed **anchor** box. Decoding inverts Eq 2: the center shifts in anchor-size units (`x = tx·wa + xa`), and the size changes in log space (`w = wa·exp(tw)`).

We also build the three aspect-ratio anchors at one scale — `w = scale·√r`, `h = scale/√r` — which all share the same area.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(0)

# A single 1:1 anchor at feature cell (1,1), stride 16, plus a predicted delta.
xa, ya, wa, ha = 24.0, 24.0, 64.0, 64.0
tx, ty, tw, th = 0.5, -0.25, 0.2, 0.0

# Decode: center shifts in anchor-size units, size changes in log space (inverse of Eq 2).
x = tx * wa + xa                                  # 0.5*64 + 24 = 56
y = ty * ha + ya                                  # -0.25*64 + 24 = 8
w = wa * math.exp(tw)                             # 64*e^0.2 = 78.17
h = ha * math.exp(th)                             # 64*e^0   = 64
print("decoded box  center=(%.2f, %.2f)  size=%.2f x %.2f" % (x, y, w, h))   # (56.00, 8.00) 78.17 x 64.00

# The 3 aspect-ratio anchors at one scale (area = 64^2 = 4096): w = scale*sqrt(r), h = scale/sqrt(r).
scale = 64.0
for r in [1.0, 0.5, 2.0]:                         # 1:1, 1:2, 2:1
    aw = scale * math.sqrt(r)
    ah = scale / math.sqrt(r)
    print("ratio %3.1f -> w=%.2f h=%.2f (area %.0f)" % (r, aw, ah, aw * ah))

### Step 2 — Place anchors everywhere and define the RPN head

The RPN places `k` anchors at **every** feature-map cell, centered on that cell's image-pixel location. The head itself is tiny: a 3×3 conv that "slides" over the feature map (Sec 3.1), then two 1×1 convs — one emitting `2k` objectness scores, one emitting `4k` box deltas per location.

In [ ]:
# Place k anchors at every feature-map cell.
def generate_anchors(Wf, Hf, stride, scales, ratios):
    anchors = []
    for i in range(Hf):
        for j in range(Wf):
            cx = (j + 0.5) * stride          # cell center in image pixels
            cy = (i + 0.5) * stride
            for s in scales:
                for r in ratios:
                    aw = s * math.sqrt(r)    # area = s^2
                    ah = s / math.sqrt(r)
                    anchors.append([cx, cy, aw, ah])   # (x_a, y_a, w_a, h_a)
    return torch.tensor(anchors)                       # (Wf*Hf*k, 4)

# The RPN head: 3x3 conv (slide) -> 1x1 cls (2k) + 1x1 reg (4k).
class RPNHead(nn.Module):
    def __init__(self, C, k, mid=64):
        super().__init__()
        self.conv = nn.Conv2d(C, mid, 3, padding=1)    # the n=3 sliding window (Sec 3.1)
        self.cls = nn.Conv2d(mid, 2 * k, 1)            # 2k objectness scores per location
        self.reg = nn.Conv2d(mid, 4 * k, 1)            # 4k box deltas per location
    def forward(self, feat):                           # feat: (B, C, Hf, Wf)
        t = F.relu(self.conv(feat))
        return self.cls(t), self.reg(t)                # (B,2k,Hf,Wf), (B,4k,Hf,Wf)

### Step 3 — Decode predicted deltas to boxes (vectorized)

Same decoding as Step 1, but applied to all anchors at once. We unbind each anchor and delta into its four columns, apply the inverse of Eq 2, and stack the results back into an `(N, 4)` box tensor.

In [ ]:
# Decode predicted deltas -> boxes (inverse of Eq 2), vectorized over N anchors.
def decode(anchors, deltas):                           # anchors,(N,4) ; deltas,(N,4)=(tx,ty,tw,th)
    xa, ya, wa, ha = anchors.unbind(1)
    tx, ty, tw, th = deltas.unbind(1)
    x = tx * wa + xa
    y = ty * ha + ya
    w = wa * torch.exp(tw)
    h = ha * torch.exp(th)
    return torch.stack([x, y, w, h], dim=1)

### Step 4 — Why multiple anchors? (k=9 vs k=1 ablation)

One feature cell faces **three** overlapping objects of different shapes: square, wide, tall. Each anchor's reg head outputs exactly **one** box. With `k=9` a near-right-shape anchor exists for each object, so all three can be fit. With `k=1` the cell has a single box output that's pulled toward square *and* wide *and* tall at once — it cannot satisfy all three (Sec 3.1.1).

We fit both and compare the summed box loss: `k=9` should drop to ~0, `k=1` should plateau high.

In [ ]:
# One feature cell facing THREE overlapping objects of different shapes.
Wf = Hf = 1
stride = 16
C = 8
feat = torch.randn(1, C, Hf, Wf)
targets = torch.tensor([[24.0, 24.0, 64.0, 64.0],      # square (1:1)
                        [24.0, 24.0, 96.0, 48.0],      # wide   (2:1)
                        [24.0, 24.0, 48.0, 96.0]])     # tall   (1:2)

def encode_target(anchor, gt):                         # Eq 2 (target deltas)
    xa, ya, wa, ha = anchor
    x, y, w, h = gt
    return torch.tensor([(x - xa) / wa, (y - ya) / ha,
                         math.log(w / wa), math.log(h / ha)])

def assign(anchors, gts):                              # each gt -> closest-aspect-ratio anchor
    idx = []
    for gt in gts:
        gr = (gt[2] / gt[3]).item()
        best = min(range(len(anchors)),
                   key=lambda a: abs((anchors[a, 2] / anchors[a, 3]).item() - gr))
        idx.append(best)
    return idx

def run(scales, ratios, steps=360):
    torch.manual_seed(0)
    k = len(scales) * len(ratios)
    anchors = generate_anchors(Wf, Hf, stride, scales, ratios)        # (k,4)
    idx = assign(anchors, targets)                                    # positive anchor per target
    t_stars = [encode_target(anchors[idx[g]], targets[g]) for g in range(len(targets))]
    head = RPNHead(C, k)
    opt = torch.optim.Adam(head.parameters(), lr=5e-3)
    for _ in range(steps):
        _, reg = head(feat)
        reg = reg.view(k, 4)                                          # (k,4)
        # box loss = sum over the positive anchors, one per target (Eq 1's reg term, p*=1 anchors)
        loss = sum(F.smooth_l1_loss(reg[idx[g]], t_stars[g]) for g in range(len(targets)))
        opt.zero_grad()
        loss.backward()
        opt.step()
    return loss.item(), idx

loss9, idx9 = run([48.0, 64.0, 96.0], [0.5, 1.0, 2.0])               # k=9
loss1, idx1 = run([64.0], [1.0])                                      # k=1 (single square) -- ABLATION
print("\nk=9: anchors-per-target", idx9, " total box loss %.4f" % loss9)
print("k=1: anchors-per-target", idx1, " total box loss %.4f" % loss1)
# k=9 assigns a distinct anchor to each shape -> loss ~0. k=1 forces all three targets onto the
# SAME single box output -> it cannot be square AND wide AND tall, so loss plateaus high (~0.06).
# (Our small toy run, not the paper's reported number.)

## Visualize the data & results

_When one feature cell faces three overlapping objects of different shapes (square, wide, tall), does an RPN with k=9 anchors fit them all with lower total box loss than the same RPN restricted to k=1 (a single square anchor, so a single box output)?_

### Step 1 — Rebuild the anchors, head, and three targets

This panel reproduces the ablation as a self-contained run that records the loss curve. We redefine the anchor generator, a reg-only RPN head, and the three differently-shaped targets at the same cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
torch.manual_seed(0)

def generate_anchors(Wf, Hf, stride, scales, ratios):
    out = []
    for i in range(Hf):
        for j in range(Wf):
            cx = (j + 0.5) * stride
            cy = (i + 0.5) * stride
            for s in scales:
                for r in ratios:
                    out.append([cx, cy, s * math.sqrt(r), s / math.sqrt(r)])
    return torch.tensor(out)

class RPNHead(nn.Module):
    def __init__(self, C, k, mid=64):
        super().__init__()
        self.conv = nn.Conv2d(C, mid, 3, padding=1)
        self.reg = nn.Conv2d(mid, 4 * k, 1)           # 4k box deltas
    def forward(self, x):
        return self.reg(F.relu(self.conv(x)))

Wf = Hf = 1
stride = 16
C = 8
feat = torch.randn(1, C, Hf, Wf)
# Three overlapping targets of different shapes at the SAME cell: square, wide, tall.
targets = torch.tensor([[24., 24., 64., 64.], [24., 24., 96., 48.], [24., 24., 48., 96.]])

def encode(anchor, gt):
    xa, ya, wa, ha = anchor
    x, y, w, h = gt
    return torch.tensor([(x - xa) / wa, (y - ya) / ha, math.log(w / wa), math.log(h / ha)])

def assign(anchors, gts):                              # each gt -> closest-aspect-ratio anchor
    out = []
    for gt in gts:
        gr = (gt[2] / gt[3]).item()
        best = min(range(len(anchors)), key=lambda a: abs((anchors[a, 2] / anchors[a, 3]).item() - gr))
        out.append(best)
    return out

### Step 2 — Fit k=9 vs k=1 and trace the loss

We fit both anchor sets and sample the summed box loss every 50 steps. With everything else identical, the only difference is the anchor set — so the gap isolates the multi-anchor design. `k=9` drives the loss to ~0 (one anchor per shape); `k=1`'s single box output can't be all three shapes and plateaus.

In [ ]:
def curve(scales, ratios, steps=360):
    torch.manual_seed(0)
    k = len(scales) * len(ratios)
    anchors = generate_anchors(Wf, Hf, stride, scales, ratios)
    idx = assign(anchors, targets)                                   # positive anchor per target
    t_stars = [encode(anchors[idx[g]], targets[g]) for g in range(len(targets))]
    head = RPNHead(C, k)
    opt = torch.optim.Adam(head.parameters(), lr=5e-3)
    hist = []
    for s in range(steps):
        reg = head(feat).view(k, 4)
        loss = sum(F.smooth_l1_loss(reg[idx[g]], t_stars[g]) for g in range(len(targets)))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if s % 50 == 0:
            hist.append(round(loss.item(), 4))
    return hist

print("k=9:", curve([48., 64., 96.], [0.5, 1., 2.]))      # -> ~0 (one anchor per shape)
print("k=1:", curve([64.], [1.]))                          # -> plateaus ~0.06 (one box can't be 3 shapes)
# k=9 assigns a distinct anchor to each shape; k=1's single box output can't fit square+wide+tall.
# Our toy run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. One feature cell faces three overlapping objects of different shapes &mdash; a
            square, a wide one, a tall one. Your RPN with $k=9$ anchors fits all three. Shrink to $k=1$ (one
            square anchor per cell, so one box output) and refit. What happens to the total box loss, and what
            does that show about why the RPN needs multiple anchors?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set the anchor set to a single $1{:}1$ square per cell ($k=1$); keep the feature map, head, optimizer, the three targets, and the seed otherwise identical. — _An honest ablation changes exactly one thing &mdash; the anchor count/shapes &mdash; so any difference is attributable to it._
- Refit and compare the summed box-regression loss over the three targets: $k=9$ drops to ~0 (each shape gets its own anchor); $k=1$ plateaus high (~0.06). — _Each anchor's reg head produces ONE box. With $k=9$ a near-right-shape anchor exists for each object; with $k=1$ the cell has a single box output that is pulled toward square, wide, AND tall at once &mdash; an average that fits none (&sect;3.1.1)._
- Conclude the multiple anchors, not extra capacity, let one location predict several differently-shaped boxes; the gap is what they are worth. — _Both runs share the head and fitting; only the anchor set differs, isolating it as the cause._

**Answer:** With $k=1$ the cell has a single box output, but three differently-shaped targets pull its
                 predicted deltas in conflicting directions &mdash; the best it can do is an average box that
                 fits none of them, so the summed smooth-L1 loss plateaus high (in our run ~0.06). With $k=9$
                 each object is assigned a distinct anchor whose shape is already close, so all three are fit and
                 the total loss falls to ~0. This is precisely why &sect;3.1.1 places $k$ anchors per location:
                 a single $1\times1$ predictor can only emit a fixed number of boxes per cell, so to detect
                 several objects of different sizes/shapes at the same position you need several anchors. Since
                 the two runs differ only in the anchor set, the gap isolates the multi-anchor design. The
                 CODEVIZ panel shows it.

</details>

**Problem 2.** A feature map is $W\times H = 50\times38$ and the RPN uses $k=9$ anchors. How many anchors are
            placed over the image? What are the output channel counts of the cls and reg $1\times1$ conv
            layers? (Per &sect;3.1 / 3.1.1.)

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Total anchors $= W\,H\,k = 50\times38\times9$. — _Each of the $WH$ feature cells gets $k$ anchors (&sect;3.1.1: "there are $WHk$ anchors in total")._
- Compute: $50\times38 = 1900$ cells; $1900\times9 = 17{,}100$ anchors. — _Just multiply the grid size by the per-cell anchor count._
- cls conv: $2k = 18$ output channels; reg conv: $4k = 36$ output channels. — _cls emits an object/background pair per anchor ($2k$); reg emits the four deltas per anchor ($4k$) (&sect;3.1)._

**Answer:** $WHk = 50\times38\times9 = 17{,}100$ anchors. The cls $1\times1$ conv has $2k = 18$ output
                 channels (object-vs-background per anchor) and the reg $1\times1$ conv has $4k = 36$ output
                 channels (the four box deltas per anchor). The spatial size of both output maps stays
                 $50\times38$ &mdash; one prediction vector per cell.

</details>

**Problem 3.** An anchor is $x_a{=}100,\,y_a{=}80,\,w_a{=}128,\,h_a{=}64$. The RPN predicts deltas
            $t=(t_x,t_y,t_w,t_h)=(0,\,0.5,\,0,\,\log 2)$. Decode the proposed box (center, width, height),
            using the inverse of Eq 2.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Center $x = t_x w_a + x_a = 0\cdot128 + 100 = 100$; $y = t_y h_a + y_a = 0.5\cdot64 + 80 = 32 + 80 = 112$. — _Center shift is in anchor-size units: $t_y=0.5$ moves down by half the anchor height (Eq 2 inverse)._
- Width $w = w_a e^{t_w} = 128\cdot e^{0} = 128$; height $h = h_a e^{t_h} = 64\cdot e^{\log 2} = 64\cdot 2 = 128$. — _Size change is in log space, so decoding exponentiates; $e^{\log 2}=2$ doubles the height._
- Report the box: center $(100,112)$, size $128\times128$. — _Combining the decoded center and size gives the proposed box._

**Answer:** Center $(x,y) = (100,\,112)$: $t_x=0$ leaves $x$ at $100$; $t_y=0.5$ shifts $y$ down by half
                 the anchor height ($0.5\cdot64=32$) to $112$. Size $w\times h = 128\times128$: $t_w=0$ keeps
                 the width at $128$; $t_h=\log 2$ doubles the height to $128$. So a $128\times64$ anchor became
                 a $128\times128$ box, shifted down by 32 pixels &mdash; the $\log/\exp$ parametrization makes
                 "double the height" a clean $t_h=\log 2$.

</details>